# Meeting Summarization with Transformer Model

This notebook trains a Transformer model on the MeetingBank dataset for automatic meeting summarization.

## Dataset: MeetingBank
- **Source**: https://huggingface.co/datasets/huuuyeah/meetingbank
- **Task**: Abstractive summarization of meeting transcripts
- **Size**: 1,366 meetings with 3,579+ hours of content
- **Segments**: 6,892 segment-level summarization instances

## Model: Transformer (Attention is All You Need)
- Encoder-Decoder architecture
- Multi-head self-attention mechanism
- Position-wise feed-forward networks
- Positional encoding for sequence information

## 1. Import Required Libraries

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter

from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace

from pathlib import Path
from tqdm import tqdm
import warnings
import json
import math

warnings.filterwarnings("ignore")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

ModuleNotFoundError: No module named 'tensorboard'

## 2. Configuration Settings

In [ ]:
def get_config():
    return {
        "batch_size": 8,  # Reduced batch size for long sequences
        "num_epochs": 20,
        "lr": 10**-4,
        "seq_len": 512,
        "max_src_len": 2048,  # Maximum length for source (transcript)
        "max_tgt_len": 512,   # Maximum length for target (summary)
        "dataset_name": "huuuyeah/meetingbank",
        "model_folder": "weights",
        "model_basename": "meeting_model_",
        "preload": None,
        "tokenizer_file": "tokenizers/meetingbank_{}.json",
        "experiment_name": "meeting_summarization",
        "vocab_size": 32000,
        "task": "summarization"
    }

def get_weights_file_path(config, epoch: str):
    model_folder = Path(config['model_folder'])
    model_basename = config['model_basename']
    model_filename = model_folder / f"{model_basename}{epoch}.pt"
    return str(Path('.') / model_folder / model_filename)

config = get_config()
print("Configuration loaded:")
for key, value in config.items():
    print(f"  {key}: {value}")

## 3. Model Components - Layer Normalization & Residual Connections

In [ ]:
class LayerNormalization(nn.Module):
    def __init__(self, eps: float = 10**-6):
        super().__init__()
        self.eps = eps
        self.alpha = nn.Parameter(torch.ones(1))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        std = x.std(dim=-1, keepdim=True)
        return self.alpha * (x - mean) / (std + self.eps) + self.bias


class ResidualConnection(nn.Module):
    def __init__(self, dropout: float):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.norm = LayerNormalization()

    def forward(self, x, sublayer):
        return x + self.dropout(sublayer(self.norm(x)))


print("✓ Layer Normalization and Residual Connection defined")

## 4. Model Components - Input Embeddings & Positional Encoding

In [ ]:
class InputEmbeddings(nn.Module):
    def __init__(self, d_model_size: int, vocab_size: int):
        super().__init__()
        self.d_model_size = d_model_size
        self.vocab_size = vocab_size
        self.embeddings = nn.Embedding(vocab_size, d_model_size)

    def forward(self, x):
        return self.embeddings(x) * math.sqrt(self.d_model_size)


class PositionalEncoding(nn.Module):
    def __init__(self, d_model_size: int, seq_len: int, dropout: float) -> None:
        super().__init__()
        self.d_model_size = d_model_size
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)

        positional_encoding = torch.zeros(seq_len, d_model_size)
        positions = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model_size, 2).float() * (-math.log(100000.0) / d_model_size))

        positional_encoding[:, 0::2] = torch.sin(positions * div_term)
        positional_encoding[:, 1::2] = torch.cos(positions * div_term)
        positional_encoding = positional_encoding.unsqueeze(0)

        self.register_buffer('pe', positional_encoding)

    def forward(self, x):
        x = x + self.pe[:, :x.shape[1], :].requires_grad_(False)
        return self.dropout(x)


print("✓ Input Embeddings and Positional Encoding defined")

## 5. Model Components - Multi-Head Attention

In [ ]:
class MultiHeadAttentionNetwork(nn.Module):
    def __init__(self, d_model_size: int, h: int, dropout: float):
        super().__init__()
        self.d_model_size = d_model_size
        self.h = h
        assert d_model_size % h == 0, "d_model_size is not divisible by h"

        self.dk = d_model_size // h
        self.wq = nn.Linear(d_model_size, d_model_size)
        self.wk = nn.Linear(d_model_size, d_model_size)
        self.wv = nn.Linear(d_model_size, d_model_size)
        self.wo = nn.Linear(d_model_size, d_model_size)
        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        d_k = query.shape[-1]
        attention_scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)
        if mask is not None:
            attention_scores.masked_fill_(mask == 0, -1e9)
        attention_scores = attention_scores.softmax(dim=-1)
        if dropout is not None:
            attention_scores = dropout(attention_scores)
        return (attention_scores @ value), attention_scores

    def forward(self, k, q, v, mask):
        query = self.wq(q)
        key = self.wk(k)
        value = self.wv(v)

        query = query.view(query.shape[0], query.shape[1], self.h, self.dk).transpose(1, 2)
        key = key.view(key.shape[0], key.shape[1], self.h, self.dk).transpose(1, 2)
        value = value.view(value.shape[0], value.shape[1], self.h, self.dk).transpose(1, 2)

        x, self.attention_scores = MultiHeadAttentionNetwork.attention(query, key, value, mask, self.dropout)
        x = x.transpose(1, 2).contiguous().view(x.shape[0], -1, self.h * self.dk)

        return self.wo(x)


print("✓ Multi-Head Attention Network defined")

## 6. Model Components - Feed Forward Network & Projection Layer

In [ ]:
class FeedForwardNetwork(nn.Module):
    def __init__(self, d_model_size: int, d_ff: int, dropout: float):
        super().__init__()
        self.linear_layer_1 = nn.Linear(d_model_size, d_ff)
        self.dropout = nn.Dropout(dropout)
        self.linear_layer_2 = nn.Linear(d_ff, d_model_size)

    def forward(self, x):
        return self.linear_layer_2(self.dropout(torch.relu(self.linear_layer_1(x))))


class ProjectionLayer(nn.Module):
    def __init__(self, d_model_size: int, vocab_size: int):
        super().__init__()
        self.d_model_size = d_model_size
        self.vocab_size = vocab_size
        self.proj = nn.Linear(d_model_size, vocab_size)

    def forward(self, x):
        return torch.log_softmax(self.proj(x), dim=-1)


print("✓ Feed Forward Network and Projection Layer defined")

## 7. Model Components - Encoder & Decoder Blocks

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, self_attention_block: MultiHeadAttentionNetwork, feed_forward_block: FeedForwardNetwork, dropout: float):
        super().__init__()
        self.self_attention_block = self_attention_block
        self.feed_forward_block = feed_forward_block
        self.residual_connection = nn.ModuleList([ResidualConnection(dropout) for _ in range(2)])

    def forward(self, x, src_mask):
        x = self.residual_connection[0](x, lambda x: self.self_attention_block(x, x, x, src_mask))
        x = self.residual_connection[1](x, self.feed_forward_block)
        return x


class Encoder(nn.Module):
    def __init__(self, layers: nn.ModuleList):
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization()

    def forward(self, x, src_mask):
        for layer in self.layers:
            x = layer(x, src_mask)
        return self.norm(x)


class DecoderBlock(nn.Module):
    def __init__(self, self_attention_block: MultiHeadAttentionNetwork, cross_attention_block: MultiHeadAttentionNetwork,
                 feed_forward_block: FeedForwardNetwork, dropout: float):
        super().__init__()
        self.self_attention_block = self_attention_block
        self.cross_attention_block = cross_attention_block
        self.feed_forward_network = feed_forward_block
        self.residual_connection = nn.ModuleList([ResidualConnection(dropout) for _ in range(3)])

    def forward(self, x, encoder_output, src_mask, target_mask):
        x = self.residual_connection[0](x, lambda x: self.self_attention_block(x, x, x, target_mask))
        x = self.residual_connection[1](x, lambda x: self.cross_attention_block(x, encoder_output, encoder_output, src_mask))
        x = self.residual_connection[2](x, self.feed_forward_network)
        return x


class Decoder(nn.Module):
    def __init__(self, layers: nn.ModuleList):
        super().__init__()
        self.layers = layers
        self.norm = LayerNormalization()

    def forward(self, x, encoder_output, src_mask, target_mask):
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, target_mask)
        return self.norm(x)


print("✓ Encoder and Decoder Blocks defined")

## 8. Complete Transformer Model

In [ ]:
class Transformer(nn.Module):
    def __init__(self, encoder: Encoder, decoder: Decoder, src_embed: InputEmbeddings,
                 target_embed: InputEmbeddings, src_pos: PositionalEncoding,
                 target_pos: PositionalEncoding, projection_layer: ProjectionLayer):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_emb = src_embed
        self.target_emb = target_embed
        self.src_pos = src_pos
        self.target_pos = target_pos
        self.projection_layer = projection_layer

    def encode(self, src, src_mask):
        src = self.src_emb(src)
        src = self.src_pos(src)
        return self.encoder(src, src_mask)

    def decode(self, encoder_output, src_mask, target, target_mask):
        target = self.target_emb(target)
        target = self.target_pos(target)
        return self.decoder(target, encoder_output, src_mask, target_mask)

    def project(self, x):
        return self.projection_layer(x)


def build_transformer(src_vocab_size: int, target_vocab_size: int, src_seq_len: int,
                     target_seq_len: int, d_model_size: int = 512, d_ff: int = 2048,
                     h: int = 8, dropout: float = 0.1, n: int = 6) -> Transformer:
    """Build a complete transformer model."""
    src_embed = InputEmbeddings(d_model_size, src_vocab_size)
    target_embed = InputEmbeddings(d_model_size, target_vocab_size)

    src_pos = PositionalEncoding(d_model_size, src_seq_len, dropout)
    target_pos = PositionalEncoding(d_model_size, target_seq_len, dropout)

    # Build encoder blocks
    encoder_blocks = []
    for _ in range(n):
        encoder_self_attention_block = MultiHeadAttentionNetwork(d_model_size, h, dropout)
        encoder_ff = FeedForwardNetwork(d_model_size, d_ff, dropout)
        encoder_block = EncoderBlock(encoder_self_attention_block, encoder_ff, dropout)
        encoder_blocks.append(encoder_block)

    # Build decoder blocks
    decoder_blocks = []
    for _ in range(n):
        decoder_self_attention_block = MultiHeadAttentionNetwork(d_model_size, h, dropout)
        decoder_cross_attention_block = MultiHeadAttentionNetwork(d_model_size, h, dropout)
        decoder_ff = FeedForwardNetwork(d_model_size, d_ff, dropout)
        decoder_block = DecoderBlock(decoder_self_attention_block, decoder_cross_attention_block, decoder_ff, dropout)
        decoder_blocks.append(decoder_block)

    encoder = Encoder(nn.ModuleList(encoder_blocks))
    decoder = Decoder(nn.ModuleList(decoder_blocks))

    projection_layer = ProjectionLayer(d_model_size, target_vocab_size)

    transformer = Transformer(encoder, decoder, src_embed, target_embed, src_pos, target_pos, projection_layer)

    # Initialize parameters with Xavier uniform
    for p in transformer.parameters():
        if p.dim() > 1:
            nn.init.xavier_uniform_(p)

    return transformer


print("✓ Complete Transformer Model defined")

## 9. Dataset Preparation - Causal Mask & Dataset Class

In [ ]:
def causal_mask(size):
    """Create causal mask to prevent attention to future positions."""
    mask = torch.triu(torch.ones(1, size, size), diagonal=1).type(torch.int)
    return mask == 0


class MeetingSummarizationDataset(Dataset):
    """Dataset for meeting summarization task."""

    def __init__(self, dataset, tokenizer_src, tokenizer_tgt, seq_len_src, seq_len_tgt):
        self.dataset = dataset
        self.tokenizer_src = tokenizer_src  # For transcripts
        self.tokenizer_tgt = tokenizer_tgt  # For summaries
        self.seq_len_src = seq_len_src
        self.seq_len_tgt = seq_len_tgt

        self.src_sos_token = torch.tensor([tokenizer_src.token_to_id('[SOS]')], dtype=torch.int64)
        self.src_eos_token = torch.tensor([tokenizer_src.token_to_id('[EOS]')], dtype=torch.int64)
        self.src_pad_token = torch.tensor([tokenizer_src.token_to_id('[PAD]')], dtype=torch.int64)

        self.tgt_sos_token = torch.tensor([tokenizer_tgt.token_to_id('[SOS]')], dtype=torch.int64)
        self.tgt_eos_token = torch.tensor([tokenizer_tgt.token_to_id('[EOS]')], dtype=torch.int64)
        self.tgt_pad_token = torch.tensor([tokenizer_tgt.token_to_id('[PAD]')], dtype=torch.int64)

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        item = self.dataset[index]
        src_text = item['transcript']
        tgt_text = item['summary']

        enc_input_tokens = self.tokenizer_src.encode(src_text).ids
        dec_input_tokens = self.tokenizer_tgt.encode(tgt_text).ids

        # Truncate if too long
        if len(enc_input_tokens) > self.seq_len_src - 2:
            enc_input_tokens = enc_input_tokens[:self.seq_len_src - 2]

        if len(dec_input_tokens) > self.seq_len_tgt - 2:
            dec_input_tokens = dec_input_tokens[:self.seq_len_tgt - 2]

        enc_num_paddings = self.seq_len_src - len(enc_input_tokens) - 2
        dec_num_paddings = self.seq_len_tgt - len(dec_input_tokens) - 1

        encoder_input = torch.cat([
            self.src_sos_token,
            torch.tensor(enc_input_tokens, dtype=torch.int64),
            self.src_eos_token,
            torch.tensor([self.src_pad_token] * enc_num_paddings, dtype=torch.int64)
        ])

        decoder_input = torch.cat([
            self.tgt_sos_token,
            torch.tensor(dec_input_tokens, dtype=torch.int64),
            torch.tensor([self.tgt_pad_token] * dec_num_paddings, dtype=torch.int64)
        ])

        label = torch.cat([
            torch.tensor(dec_input_tokens, dtype=torch.int64),
            self.tgt_eos_token,
            torch.tensor([self.tgt_pad_token] * dec_num_paddings, dtype=torch.int64)
        ])

        return {
            'encoder_input': encoder_input,
            'decoder_input': decoder_input,
            'encoder_mask': (encoder_input != self.src_pad_token).unsqueeze(0).unsqueeze(0).int(),
            'decoder_mask': (decoder_input != self.tgt_pad_token).unsqueeze(0).unsqueeze(0).int() & causal_mask(decoder_input.size(0)),
            'label': label,
            'src_text': src_text,
            'tgt_text': tgt_text,
            'id': item['id'],
            'uid': item['uid']
        }


print("✓ Causal Mask and Dataset Class defined")

## 10. Tokenizer Training & Data Loading Functions

In [ ]:
def get_all_sentences(ds, field):
    """Extract sentences from dataset for tokenizer training."""
    sentences = []
    for example in ds:
        sentences.append(example[field])
    return sentences


def get_or_build_tokenizer(config, dataset, field):
    """Build or load tokenizer for meeting summarization."""
    tokenizer_file = Path(config['tokenizer_file'].format(field))
    tokenizer_file.parent.mkdir(parents=True, exist_ok=True)

    if tokenizer_file.exists():
        print(f"Loading existing tokenizer from {tokenizer_file}")
        tokenizer = Tokenizer.from_file(str(tokenizer_file))
    else:
        print(f"Training new tokenizer for {field}...")
        tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
        tokenizer.pre_tokenizer = Whitespace()
        trainer = WordLevelTrainer(
            vocab_size=config['vocab_size'],
            special_tokens=["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]", "[SOS]", "[POS]", "[EOS]"],
            min_frequency=2
        )
        tokenizer.train_from_iterator(get_all_sentences(dataset, field), trainer=trainer)
        tokenizer.save(str(tokenizer_file))
        print(f"Tokenizer saved to {tokenizer_file}")

    return tokenizer


def save_dataset_to_json(dataset, filename='meetingbank_dataset.json'):
    """Save dataset samples to JSON file for inspection."""
    with open(filename, 'w') as f:
        for i, item in enumerate(dataset):
            if i >= 10:
                break
            sample = {
                'id': item['id'],
                'uid': item['uid'],
                'summary': item['summary'][:200] + '...' if len(item['summary']) > 200 else item['summary'],
                'transcript': item['transcript'][:500] + '...' if len(item['transcript']) > 500 else item['transcript']
            }
            f.write(json.dumps(sample) + '\n')
    print(f"Dataset samples saved to {filename}")


print("✓ Tokenizer and data loading functions defined")

## 11. Load MeetingBank Dataset

In [ ]:
def get_dataset(config):
    """Load MeetingBank dataset and prepare dataloaders."""
    print(f"Loading dataset: {config['dataset_name']}")
    meetingbank = load_dataset(config['dataset_name'])

    train_raw = meetingbank['train']
    val_raw = meetingbank['validation']
    test_raw = meetingbank['test']

    print(f"Train size: {len(train_raw)}, Val size: {len(val_raw)}, Test size: {len(test_raw)}")

    # Save sample to JSON for inspection
    save_dataset_to_json(train_raw)

    # Build tokenizers for transcript and summary
    src_tokenizer = get_or_build_tokenizer(config, train_raw, 'transcript')
    tgt_tokenizer = get_or_build_tokenizer(config, train_raw, 'summary')

    print(f"Source vocab size: {src_tokenizer.get_vocab_size()}")
    print(f"Target vocab size: {tgt_tokenizer.get_vocab_size()}")

    # Create dataset objects
    train_dataset = MeetingSummarizationDataset(
        train_raw,
        src_tokenizer,
        tgt_tokenizer,
        config['max_src_len'],
        config['max_tgt_len']
    )
    val_dataset = MeetingSummarizationDataset(
        val_raw,
        src_tokenizer,
        tgt_tokenizer,
        config['max_src_len'],
        config['max_tgt_len']
    )

    # Calculate max lengths in dataset (sample)
    max_len_src = 0
    max_len_tgt = 0

    print("Calculating max sequence lengths (sampling first 1000)...")
    for i, item in enumerate(train_raw):
        if i >= 1000:
            break
        src_ids = src_tokenizer.encode(item['transcript']).ids
        tgt_ids = tgt_tokenizer.encode(item['summary']).ids
        max_len_src = max(max_len_src, len(src_ids))
        max_len_tgt = max(max_len_tgt, len(tgt_ids))

    print(f"Max length of transcript (sampled): {max_len_src}")
    print(f"Max length of summary (sampled): {max_len_tgt}")

    train_dataloader = DataLoader(train_dataset, batch_size=config['batch_size'], shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=1, shuffle=True)

    return train_dataloader, val_dataloader, src_tokenizer, tgt_tokenizer


print("✓ Dataset loading function defined")

## 12. Prepare Data and Tokenizers

**Note:** This step will download the MeetingBank dataset and train tokenizers if they don't exist. This may take several minutes.

In [ ]:
# Load dataset and prepare dataloaders
train_dataloader, val_dataloader, src_tokenizer, tgt_tokenizer = get_dataset(config)

print(f"\n✓ Dataset prepared successfully!")
print(f"Training batches: {len(train_dataloader)}")
print(f"Validation batches: {len(val_dataloader)}")

## 13. Initialize Model and Training Components

In [ ]:
# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Create model folder
Path(config['model_folder']).mkdir(parents=True, exist_ok=True)

# Build transformer model
model = build_transformer(
    src_vocab_size=src_tokenizer.get_vocab_size(),
    target_vocab_size=tgt_tokenizer.get_vocab_size(),
    src_seq_len=config['max_src_len'],
    target_seq_len=config['max_tgt_len']
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Initialize optimizer and loss function
optimizer = torch.optim.Adam(model.parameters(), lr=config['lr'], eps=1e-9)
loss_fn = nn.CrossEntropyLoss(ignore_index=src_tokenizer.token_to_id('[PAD]'), label_smoothing=0.1)

# Initialize TensorBoard writer
writer = SummaryWriter(log_dir=f"runs/{config['experiment_name']}")

print("\n✓ Model and training components initialized!")

## 14. Training Loop Function

In [ ]:
def train_model(config, model, train_dataloader, val_dataloader, optimizer, loss_fn, writer, device, src_tokenizer, tgt_tokenizer):
    """Train the transformer model."""

    initial_epoch = 0
    global_step = 0

    # Load checkpoint if preload is specified
    if config['preload'] is not None:
        model_filename = get_weights_file_path(config, config['preload'])
        print(f"Loading model from {model_filename}")
        state = torch.load(model_filename)
        model.load_state_dict(state['model_state_dict'])
        initial_epoch = state['epoch'] + 1
        optimizer.load_state_dict(state['optimizer'])
        global_step = state['global_step']

    # Training loop
    for epoch in range(initial_epoch, config['num_epochs']):
        model.train()
        batch_iterator = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{config['num_epochs']}", unit="batch")

        epoch_loss = 0
        for batch_idx, batch in enumerate(batch_iterator):
            src = batch['encoder_input'].to(device)
            tgt = batch['decoder_input'].to(device)
            encoder_mask = batch['encoder_mask'].to(device)
            decoder_mask = batch['decoder_mask'].to(device)
            label = batch['label'].to(device)

            # Forward pass
            optimizer.zero_grad()
            encoder_output = model.encode(src, encoder_mask)
            decoder_output = model.decode(encoder_output, encoder_mask, tgt, decoder_mask)
            logits = model.project(decoder_output)

            # Calculate loss
            loss = loss_fn(logits.view(-1, tgt_tokenizer.get_vocab_size()), label.view(-1))
            epoch_loss += loss.item()

            # Update progress bar
            batch_iterator.set_postfix(loss=f"{loss.item():6.3f}")

            # Log to TensorBoard
            writer.add_scalar('train loss', loss.item(), global_step)
            writer.flush()

            # Backward pass
            loss.backward()
            optimizer.step()

            global_step += 1

        # Print epoch summary
        avg_epoch_loss = epoch_loss / len(train_dataloader)
        print(f"Epoch {epoch+1} - Average Loss: {avg_epoch_loss:.4f}")

        # Save checkpoint
        model_filename = get_weights_file_path(config, f'{epoch:02d}')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'global_step': global_step
        }, model_filename)
        print(f"Model saved to {model_filename}")

    print("\n✓ Training completed!")


print("✓ Training function defined")

## 15. Start Training

**Warning:** Training will take a significant amount of time depending on your hardware. With GPU, expect several hours. Without GPU, it may take much longer.

You can adjust the `num_epochs` in the configuration above to train for fewer epochs initially.

In [ ]:
# Start training
print("=" * 80)
print("STARTING TRAINING")
print("=" * 80)

train_model(
    config=config,
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    writer=writer,
    device=device,
    src_tokenizer=src_tokenizer,
    tgt_tokenizer=tgt_tokenizer
)

print("\n" + "=" * 80)
print("TRAINING FINISHED")
print("=" * 80)

## 16. Visualize Training Progress (Optional)

You can monitor training in real-time using TensorBoard:

```bash
tensorboard --logdir runs/meeting_summarization
```

Then open your browser to `http://localhost:6006`

## 17. Test Inference (After Training)

After training completes, you can test the model on a sample meeting transcript.

In [ ]:
def greedy_decode(model, source, source_mask, tokenizer_src, tokenizer_tgt, max_len, device):
    """Generate summary using greedy decoding."""
    sos_idx = tokenizer_tgt.token_to_id('[SOS]')
    eos_idx = tokenizer_tgt.token_to_id('[EOS]')

    # Encode the source
    encoder_output = model.encode(source, source_mask)

    # Initialize decoder input with SOS token
    decoder_input = torch.empty(1, 1).fill_(sos_idx).type_as(source).to(device)

    while True:
        if decoder_input.size(1) >= max_len:
            break

        # Build mask for target
        decoder_mask = causal_mask(decoder_input.size(1)).type_as(source_mask).to(device)

        # Calculate output
        out = model.decode(encoder_output, source_mask, decoder_input, decoder_mask)

        # Get next token
        prob = model.project(out[:, -1])
        _, next_word = torch.max(prob, dim=1)
        decoder_input = torch.cat([decoder_input, torch.empty(1, 1).type_as(source).fill_(next_word.item()).to(device)], dim=1)

        if next_word == eos_idx:
            break

    return decoder_input.squeeze(0)


def test_sample_inference(model, val_dataloader, src_tokenizer, tgt_tokenizer, device):
    """Test the model on a sample from validation set."""
    model.eval()

    with torch.no_grad():
        # Get a sample batch
        batch = next(iter(val_dataloader))

        source = batch['encoder_input'].to(device)
        source_mask = batch['encoder_mask'].to(device)

        # Generate summary
        model_out = greedy_decode(model, source, source_mask, src_tokenizer, tgt_tokenizer,
                                 config['max_tgt_len'], device)

        # Decode tokens to text
        source_text = batch['src_text'][0]
        target_text = batch['tgt_text'][0]
        model_out_text = tgt_tokenizer.decode(model_out.detach().cpu().numpy())

        print("\n" + "="*80)
        print("SAMPLE INFERENCE")
        print("="*80)
        print(f"\nSource (transcript excerpt):\n{source_text[:500]}...")
        print(f"\nTarget (actual summary):\n{target_text}")
        print(f"\nModel Output (generated summary):\n{model_out_text}")
        print("="*80)


print("✓ Inference functions defined")
print("\nTo test inference after training, run:")
print("test_sample_inference(model, val_dataloader, src_tokenizer, tgt_tokenizer, device)")

## Summary

This notebook provides a complete implementation of a Transformer model for meeting summarization:

### Files Generated:
- `weights/meeting_model_*.pt` - Model checkpoints
- `tokenizers/meetingbank_transcript.json` - Transcript tokenizer
- `tokenizers/meetingbank_summary.json` - Summary tokenizer
- `meetingbank_dataset.json` - Sample data for inspection
- `runs/meeting_summarization/` - TensorBoard logs

### Next Steps:
1. Monitor training with TensorBoard
2. Test inference on validation samples
3. Fine-tune hyperparameters if needed
4. Evaluate on test set using ROUGE metrics